# VoiceSwap AI Converter (free Colab GPU)
This turns on **real voice cloning** for your VoiceSwap website. The new address is sent to your website **automatically**, so you never copy it.

**One-time setup on Render:** open your VoiceSwap service → **Environment** → add `VC_ENGINE` = `http` and `CONVERTER_KEY` = a secret word you choose (for example 12 random letters). Save.

**Every time you use this notebook:**
1. Type your website address and the same secret word in the first code cell below.
2. Menu **Runtime → Change runtime type → T4 GPU → Save**.
3. Menu **Runtime → Run all**. The first time takes 5 to 10 minutes.
4. When you see `CONNECTED`, open your VoiceSwap website and try it.
5. Keep this page open while people use the app.

In [ ]:
#@title Step 1: type your two settings here, then Runtime > Run all
SITE_URL = "https://YOUR-APP.onrender.com"  #@param {type:"string"}
KEY = ""  #@param {type:"string"}
SITE_URL = SITE_URL.strip().rstrip("/")
assert SITE_URL.startswith("https://") and "YOUR-APP" not in SITE_URL, "Type your real website address in SITE_URL (it starts with https://)"
assert len(KEY.strip()) >= 8, "Type your secret word in KEY (the same one you put in Render as CONVERTER_KEY, at least 8 characters)"
KEY = KEY.strip()
print("settings ok")

In [ ]:
# Step 2: check that a GPU is on
import subprocess
out = subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout
print(out or "NO GPU FOUND. Go to Runtime > Change runtime type > T4 GPU, then Run all again.")
assert out, "No GPU"

In [ ]:
# Step 3: get Seed-VC (the AI voice model) and install it. This is the slow step.
!git clone -q https://github.com/Plachtaa/seed-vc /content/seed-vc
!cd /content/seed-vc && pip install -q -r requirements.txt
print("Seed-VC installed")

In [ ]:
%%writefile /content/server.py
"""Converter server for VoiceSwap's "http" engine. Wraps Seed-VC (https://github.com/Plachtaa/seed-vc).
Uses only the Python standard library (nothing to pip install for the server itself).

Run:  SEED_VC_DIR=/content/seed-vc CONVERTER_KEY=secret python server.py
  POST /   multipart: source (wav), target (wav), steps   ->  audio/wav
  GET  /health
"""
import glob, os, subprocess, sys, tempfile, threading
from email.parser import BytesParser
from email.policy import HTTP
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

SEED_VC_DIR = os.environ.get("SEED_VC_DIR", "/opt/seed-vc")
KEY = os.environ.get("CONVERTER_KEY", "")
PORT = int(os.environ.get("PORT", "8000"))
INFER = os.environ.get("SEED_VC_SCRIPT", "inference.py")
LOCK = threading.Lock()          # one conversion at a time (one GPU)
MAX_BODY = 60 * 1024 * 1024


def parse_multipart(ctype, body):
    msg = BytesParser(policy=HTTP).parsebytes(b"Content-Type: " + ctype.encode() + b"\r\nMIME-Version: 1.0\r\n\r\n" + body)
    parts = {}
    for p in msg.iter_parts():
        name = p.get_param("name", header="content-disposition")
        if name:
            parts[name] = p.get_payload(decode=True) or b""
    return parts


def convert(source, target, steps):
    with tempfile.TemporaryDirectory() as d:
        s, t, out = os.path.join(d, "source.wav"), os.path.join(d, "target.wav"), os.path.join(d, "out")
        open(s, "wb").write(source); open(t, "wb").write(target); os.makedirs(out)
        cmd = [sys.executable, INFER, "--source", s, "--target", t, "--output", out, "--diffusion-steps", str(max(4, min(steps, 50))),
               "--length-adjust", "1.0", "--inference-cfg-rate", "0.7", "--f0-condition", "False", "--auto-f0-adjust", "False",
               "--fp16", "True" if os.environ.get("FP16", "1") == "1" else "False"]
        r = subprocess.run(cmd, cwd=SEED_VC_DIR, capture_output=True, text=True, timeout=900)
        wavs = sorted(glob.glob(os.path.join(out, "*.wav")), key=os.path.getmtime)
        if r.returncode != 0 or not wavs:
            raise RuntimeError(((r.stderr or r.stdout or "no output")[-300:]).strip())
        return open(wavs[-1], "rb").read()


class H(BaseHTTPRequestHandler):
    def _send(self, code, body, ctype="application/json"):
        if isinstance(body, str): body = body.encode()
        self.send_response(code); self.send_header("Content-Type", ctype); self.send_header("Content-Length", str(len(body))); self.end_headers(); self.wfile.write(body)

    def do_GET(self):
        self._send(200, '{"ok":true}') if self.path.startswith("/health") else self._send(404, '{"error":"not found"}')

    def do_POST(self):
        try:
            n = int(self.headers.get("Content-Length", "0"))
            if n <= 0 or n > MAX_BODY:
                return self._send(413, '{"error":"bad size"}')
            body = self.rfile.read(n)                    # read first, so the client is never cut off mid-upload
            if KEY and self.headers.get("Authorization", "") != "Bearer " + KEY:
                return self._send(401, '{"error":"bad key"}')
            parts = parse_multipart(self.headers.get("Content-Type", ""), body)
            if not parts.get("source") or not parts.get("target"):
                return self._send(400, '{"error":"send source and target"}')
            steps = int((parts.get("steps") or b"25").decode() or 25)
            with LOCK:
                wav = convert(parts["source"], parts["target"], steps)
            self._send(200, wav, "audio/wav")
        except Exception as e:                       # never crash the server
            self._send(500, ('{"error":"%s"}' % str(e).replace('"', "'").replace("\n", " ")[:300]))

    def log_message(self, *a):
        pass


if __name__ == "__main__":
    print(f"Converter listening on port {PORT}, Seed-VC folder: {SEED_VC_DIR}", flush=True)
    ThreadingHTTPServer(("0.0.0.0", PORT), H).serve_forever()

In [ ]:
# Step 4: start the converter
import os, subprocess, time, urllib.request, sys
env = dict(os.environ, SEED_VC_DIR="/content/seed-vc", CONVERTER_KEY=KEY, PORT="8000")
srv = subprocess.Popen([sys.executable, "/content/server.py"], env=env)
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/health"); break
    except Exception: time.sleep(1)
print("converter is running")

In [ ]:
# Step 5: make a public https address (free Cloudflare tunnel) and tell your website about it
import re, json, threading
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
tun = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"], stderr=subprocess.PIPE, text=True)
URL = None
for line in tun.stderr:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m: URL = m.group(0); break
threading.Thread(target=lambda: [None for _ in tun.stderr], daemon=True).start()

def register():
    req = urllib.request.Request(SITE_URL + "/api/converter/register", data=json.dumps({"url": URL + "/"}).encode(), method="POST",
        headers={"Authorization": "Bearer " + KEY, "Content-Type": "application/json", "User-Agent": "VoiceSwap-Colab"})
    try:
        urllib.request.urlopen(req, timeout=90); return None
    except urllib.error.HTTPError as e:
        return f"website said {e.code}: " + e.read().decode()[:150]
    except Exception as e:
        return str(e)[:150]

err = None
for _ in range(4):                      # a sleeping free Render site needs a minute to wake up
    err = register()
    if not err: break
    time.sleep(15)
print("\n=========================================")
print("CONNECTED to your website" if not err else "NOT CONNECTED: " + str(err))
print("Converter address (for your information):", URL)
print("=========================================")
if err: print("Check SITE_URL, and that Render has CONVERTER_KEY set to the same secret word and VC_ENGINE=http.")

In [ ]:
# Step 6: keep running. Every minute it tells your website the address again (this also keeps a free Render site awake).
last = None
while True:
    time.sleep(60)
    e = register()
    if e != last:
        print("website link:", "ok" if not e else "problem - " + e); last = e